In [1]:
import sys
from pathlib import Path

project_root = Path("..").resolve()
sys.path.append(str(project_root))

In [2]:
import pandas as pd
from src.features import compute_rsi, compute_moving_average, compute_volatility
from src.model import time_series_split, train_logistic_regression, train_random_forest, evaluate_model




df = pd.read_csv("../data/raw/AAPL.csv", index_col=0)

# FIX: ensure Close is numeric
df["Close"] = pd.to_numeric(df["Close"], errors="coerce")

df["rsi_14"] = compute_rsi(df["Close"], 14)
df["ma_10"] = compute_moving_average(df["Close"], 10)
df["ma_20"] = compute_moving_average(df["Close"], 20)
df["volatility_10"] = compute_volatility(df["Close"], 10)

df.head(20)


,Close,High,Low,Open,Volume,rsi_14,ma_10,ma_20,volatility_10
Price,,,,,,,,,
Ticker,NaN,AAPL,AAPL,AAPL,AAPL,NaN,NaN,NaN,NaN
Date,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2024-01-02,183.903244,186.67705163638013,182.16961614691823,185.3991117688797,82488700,NaN,NaN,NaN,NaN
2024-01-03,182.526230,184.1409850600839,181.7138941984477,182.49651173581816,58414500,NaN,NaN,NaN,NaN
2024-01-04,180.208115,181.37706768373354,179.18775216816366,180.4458595901839,71983600,NaN,NaN,NaN,NaN
2024-01-05,179.484940,181.05015949262307,178.48439420396812,180.28737421180142,62379700,NaN,NaN,NaN,NaN
2024-01-08,183.823975,183.86360885082058,179.80196071659407,180.38643729162627,59144500,NaN,NaN,NaN,NaN
2024-01-09,183.407913,183.41781421075365,181.0204564512382,182.19932576748687,42841800,NaN,NaN,NaN,NaN
2024-01-10,184.448074,184.65610118978807,182.19930742900604,182.62529236075727,46792900,NaN,NaN,NaN,NaN


In [3]:
df["target"] = (df["Close"].shift(-1) > df["Close"]).astype(int)

df[["Close", "target"]].head(10)

,Close,target
Price,,
Ticker,NaN,0
Date,NaN,0
2024-01-02,183.903244,0
2024-01-03,182.526230,0
2024-01-04,180.208115,0
2024-01-05,179.484940,1
2024-01-08,183.823975,0
2024-01-09,183.407913,1
2024-01-10,184.448074,0


In [4]:
df_final = df.dropna().copy()


feature_cols = ["rsi_14", "ma_10", "ma_20", "volatility_10"]
X = df_final[feature_cols]
y = df_final["target"]

In [5]:
X = df_final[feature_cols]
y = df_final["target"]


In [6]:
X.head()

,rsi_14,ma_10,ma_20,volatility_10
Price,,,,
2024-01-30,55.832606,189.490460,186.132671,0.014680
2024-01-31,46.739534,189.660852,186.071249,0.016116
2024-02-01,52.166515,189.485507,186.200529,0.012728
2024-02-02,49.883310,188.919852,186.395687,0.011381
2024-02-05,56.857444,188.304659,186.717647,0.011058


In [7]:
y.head()

Price
2024-01-30    0
2024-01-31    1
2024-02-01    0
2024-02-02    1
2024-02-05    1
Name: target, dtype: int64

In [8]:
y.value_counts(normalize=True)


target
1    0.546748
0    0.453252
Name: proportion, dtype: float64

In [9]:
X_train, X_test, y_train, y_test = time_series_split(X, y)

model, scaler = train_logistic_regression(X_train, y_train)

accuracy, report = evaluate_model(
    model,
    scaler,
    X_test,
    y_test
)

print("Accuracy:", accuracy)
print("\nClassification Report:\n")
print(report)


Accuracy: 0.5454545454545454

Classification Report:

              precision    recall  f1-score   support

           0       0.51      0.39      0.44        46
           1       0.56      0.68      0.62        53

    accuracy                           0.55        99
   macro avg       0.54      0.54      0.53        99
weighted avg       0.54      0.55      0.54        99



In [10]:
X_train.shape, X_test.shape


((393, 4), (99, 4))

In [11]:
y_train.shape, y_test.shape


((393,), (99,))

In [12]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(max_iter=1000)
model.fit(X_train, y_train)


LogisticRegression(max_iter=1000)

In [13]:
y_pred = model.predict(X_test)

In [14]:
from sklearn.metrics import accuracy_score, classification_report

accuracy = accuracy_score(y_test, y_pred)
print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.5858585858585859

Classification Report:

              precision    recall  f1-score   support

           0       0.56      0.54      0.55        46
           1       0.61      0.62      0.62        53

    accuracy                           0.59        99
   macro avg       0.58      0.58      0.58        99
weighted avg       0.59      0.59      0.59        99



In [15]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

In [16]:
from sklearn.linear_model import LogisticRegression

model_scaled = LogisticRegression(max_iter=1000)
model_scaled.fit(X_train_scaled, y_train)

LogisticRegression(max_iter=1000)

In [17]:
y_pred_scaled = model_scaled.predict(X_test_scaled)

from sklearn.metrics import accuracy_score, classification_report

print("Scaled Accuracy:", accuracy_score(y_test, y_pred_scaled))
print("\nClassification Report:\n")
print(classification_report(y_test, y_pred_scaled))


Scaled Accuracy: 0.5454545454545454

Classification Report:

              precision    recall  f1-score   support

           0       0.51      0.39      0.44        46
           1       0.56      0.68      0.62        53

    accuracy                           0.55        99
   macro avg       0.54      0.54      0.53        99
weighted avg       0.54      0.55      0.54        99



In [18]:
# Time-based split (same split for fair comparison)
X_train, X_test, y_train, y_test = time_series_split(X, y)

# Train RandomForest
rf_model = train_random_forest(X_train, y_train)

# Evaluate RandomForest
from sklearn.metrics import accuracy_score, classification_report

y_pred_rf = rf_model.predict(X_test)

print("RandomForest Accuracy:", accuracy_score(y_test, y_pred_rf))
print("\nRandomForest Classification Report:\n")
print(classification_report(y_test, y_pred_rf))


RandomForest Accuracy: 0.41414141414141414

RandomForest Classification Report:

              precision    recall  f1-score   support

           0       0.42      0.70      0.52        46
           1       0.39      0.17      0.24        53

    accuracy                           0.41        99
   macro avg       0.41      0.43      0.38        99
weighted avg       0.41      0.41      0.37        99



In [30]:
import joblib

joblib.dump(model, "../models/logistic_model.pkl")
joblib.dump(scaler, "../models/scaler.pkl")

['../models/scaler.pkl']

In [31]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import joblib
from pathlib import Path

# -----------------------------
# Rebuild model as a pipeline
# -----------------------------
pipeline = Pipeline(
    steps=[
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(max_iter=1000))
    ]
)

pipeline.fit(X_train, y_train)

# -----------------------------
# Save pipeline
# -----------------------------
PROJECT_ROOT = Path.cwd().parent
MODELS_DIR = PROJECT_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)

joblib.dump(pipeline, MODELS_DIR / "logistic_pipeline.pkl")

print("Pipeline saved successfully")


Pipeline saved successfully
